# Vaihe 1: Datan haku Binancesta

Hakee 2024-01-01 – 2026-05-08 OHLCV-dataa kahdelle symbolille kolmella
aikakehyksellä ja tallentaa Parquet-tiedostoiksi `data/raw/`-kansioon.

- **Symbolit:** ETHUSDT, SOLUSDT
- **Aikakehykset:** 15min (pää), 1h ja 4h (multi-timeframe-piirteille)
- **Aikaväli:** 1.1.2024 – 8.5.2026 (kiinnitetty toistettavuuden vuoksi)

In [ ]:
# Lataa src/-moduulit uudelleen automaattisesti, jos editoit niitä
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from datetime import datetime, timezone

# Lisää projektin juuri Python-polkuun, jotta src/-tuonti toimii notebookista
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data_fetcher import fetch_and_save

print(f"Projektin juuri: {project_root}")

In [ ]:
SYMBOLS = ["ETHUSDT", "SOLUSDT"]
INTERVALS = ["15m", "1h", "4h"]

# Aikaväli kiinnitetty toistettavuuden vuoksi (ks. raportin luku 2.3)
START = datetime(2024, 1, 1, tzinfo=timezone.utc)
END = datetime(2026, 5, 8, tzinfo=timezone.utc)

print(f"Aikaväli: {START.date()} – {END.date()}")
print(f"Symbolit: {SYMBOLS}")
print(f"Aikakehykset: {INTERVALS}")
print(f"Yhteensä haetaan: {len(SYMBOLS) * len(INTERVALS)} tiedostoa")

In [ ]:
output_dir = project_root / "data" / "raw"

for symbol in SYMBOLS:
    print(f"\n{symbol}")
    for interval in INTERVALS:
        fetch_and_save(symbol, interval, START, END, output_dir=output_dir)

print("\n✅ Kaikki haut valmiit!")

In [ ]:
import pandas as pd

# Lataa ETHUSDT 15min ja katso miltä se näyttää
df = pd.read_parquet(output_dir / "ETHUSDT_15m.parquet")

print(f"Rivejä: {len(df):,}")
print(f"Aikaväli: {df.index.min()} – {df.index.max()}")
print(f"Sarakkeet: {list(df.columns)}")
print(f"\nEnsimmäiset 3 riviä:")
df.head(3)

In [ ]:
# Onko data tasaisesti 15 minuutin välein, vai onko aukkoja?
time_diffs = df.index.to_series().diff().dropna()
expected = pd.Timedelta(minutes=15)

print(f"Tyypillinen aikaero: {time_diffs.mode().iloc[0]} (odotettu: {expected})")
print(f"Aukkoja (aikaero > 15min): {(time_diffs > expected).sum()}")
print(f"\nPoikkeavat aikaerot (top 5):")
print(time_diffs[time_diffs != expected].value_counts().head())

In [ ]:
# Sanity-tarkistukset numeerisille arvoille
print("NaN-arvoja sarakkeittain:")
print(df[["open", "high", "low", "close", "volume"]].isna().sum())
print()
print(f"Negatiivisia tai nollahintoja: {((df[['open','high','low','close']] <= 0).any(axis=1)).sum()}")
print(f"Rivejä joissa high < low (mahdoton): {(df['high'] < df['low']).sum()}")
print(f"Negatiivisia volyymejä: {(df['volume'] < 0).sum()}")

In [ ]:
print(f"{'Tiedosto':<25} {'Rivejä':>10} {'Alku':>12} {'Loppu':>12} {'Koko (KB)':>12}")
print("-" * 75)

for path in sorted(output_dir.glob("*.parquet")):
    df_check = pd.read_parquet(path)
    size_kb = path.stat().st_size / 1024
    print(f"{path.name:<25} {len(df_check):>10,} "
          f"{str(df_check.index.min().date()):>12} "
          f"{str(df_check.index.max().date()):>12} "
          f"{size_kb:>12.1f}")